In [100]:
import pandas as pd
from qiskit.quantum_info import SparsePauliOp, Statevector, Pauli
from qiskit.circuit.library import PauliEvolutionGate
import numpy as np
from scipy.linalg import expm
import multiprocessing as mp
from dataclasses import dataclass
import numpy as np
from te_pai import pai, sampling
from qulacs import QuantumCircuit, Observable, QuantumState
from qulacs.gate import X, PauliRotation, DenseMatrix
from scipy.sparse.linalg import expm_multiply

In [ ]:
# Import Hamiltonian data
# This is rightmost is qubit 0 (LSB)
df = pd.read_csv("ES_H4_linear_R1.2_sto-6g.csv")
coeffs = df["coef"].astype(float).tolist()
paulis = df["label"].tolist()
nq = len(paulis[0]) if paulis else 0

In [185]:
# initial Hartree-Fock state
psi0 = QuantumState(nq)
psi0.set_computational_basis(int("00001111", 2))
T = 1.0
n_snaps = 10
N = 100  # Trotter steps

In [ ]:
def label_to_qulacs_str(label: str) -> str:
    terms = []
    for q in range(nq):
        p = label[nq-1-q]
        if p != "I":
            terms.append(f"{p} {q}")
    return " ".join(terms)

H = Observable(nq)
for lab, c in zip(paulis, coeffs):
    H.add_operator(c, label_to_qulacs_str(lab))

In [177]:
Hmat = H.get_matrix()
psiT = expm_multiply((-1j*T) * Hmat, psi0.get_vector()) 

def z_expect_from_vec(vec, q):
    prob = np.abs(vec)**2
    bits = (np.arange(prob.size) >> q) & 1  
    return float(np.sum((1 - 2*bits) * prob))

def site_occupations_from_vec(vec, n_qubits):
    occs = []
    for i in range(n_qubits // 2):
        q_up, q_dn = 2*i, 2*i+1
        n_up = 0.5 * (1 - z_expect_from_vec(vec, q_up))
        n_dn = 0.5 * (1 - z_expect_from_vec(vec, q_dn))
        occs.append(n_up + n_dn)
    return occs

occs = site_occupations_from_vec(psiT, nq)
for i, ni in enumerate(occs):
    print(f"site {i}: n_i = {ni:.6f}")

site 0: n_i = 1.930536
site 1: n_i = 1.912186
site 2: n_i = 0.089344
site 3: n_i = 0.067935


In [178]:
H = SparsePauliOp.from_list(list(zip(paulis, coeffs)))
sv0 = Statevector.from_label("00001111") 

U = PauliEvolutionGate(H, time=T, synthesis=MatrixExponential())
svT = sv0.evolve(U)

def z_expect_qiskit(sv, q, nq):
    s = ['I'] * nq
    s[nq - 1 - q] = 'Z'
    return sv.expectation_value(Pauli(''.join(s))).real

def site_occupations_qiskit(sv, nq):
    occs = []
    for i in range(nq // 2):
        q_up, q_dn = 2*i, 2*i+1
        n_up = 0.5 * (1 - z_expect_qiskit(sv, q_up, nq))
        n_dn = 0.5 * (1 - z_expect_qiskit(sv, q_dn, nq))
        occs.append(n_up + n_dn)
    return occs

occs = site_occupations_qiskit(svT, nq)
for i, ni in enumerate(occs):
    print(f"  i={i}: n_i={ni:.6f}")

  i=0: n_i=1.930536
  i=1: n_i=1.912186
  i=2: n_i=0.089344
  i=3: n_i=0.067935


In [ ]:
PMAP_ID = {"I":0, "X":1, "Y":2, "Z":3}

def label_to_indices_ids(label):
    n = len(label)
    idx, ids = [], []
    for q in range(n):
        p = label[n-1-q]
        if p != "I":
            idx.append(q)
            ids.append(PMAP_ID[p])
    return idx, ids

def build_trotter_circuit(labels, coeffs, T, N):
    dt = T / N
    circ = QuantumCircuit(len(labels[0]))
    for _ in range(N):
        for lab, c in zip(labels, coeffs):
            idx, ids = label_to_indices_ids(lab)
            if not idx:
                continue
            angle = 2.0 * dt * c
            circ.add_gate(PauliRotation(idx, ids, angle))
    return circ

U_circ = build_trotter_circuit(paulis, coeffs, T, N)


psi = QuantumState(nq)
psi.set_computational_basis(int("00001111", 2))
U_circ.update_quantum_state(psi)

def z_expect_qulacs(state, q):
    obs = Observable(nq)
    obs.add_operator(1.0, f"Z {q}")
    return float(obs.get_expectation_value(state).real)

def site_occupations_qulacs(state):
    occs = []
    for i in range(nq // 2):
        q_up, q_dn = 2*i, 2*i+1
        n_up = 0.5 * (1 - z_expect_qulacs(state, q_up))
        n_dn = 0.5 * (1 - z_expect_qulacs(state, q_dn))
        occs.append(n_up + n_dn)
    return occs

occs = site_occupations_qulacs(psi)
for i, ni in enumerate(occs):
    print(f"site {i}: n_i = {ni:.6f}")

site 0: n_i = 1.930539
site 1: n_i = 1.912182
site 2: n_i = 0.089345
site 3: n_i = 0.067933
